# Shoe Purchase Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.11


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.


## Problem Statement

Zapato a tu medida wants to buy shoe boxes from two wholesalers, Nikis and Pinki, with a budget of
`50,000 USD`. The company wants the same number of boxes from each supplier, at least `250` pairs
of shoes in total, and the printed model in section `2.11.1` imposes a minimum purchase of `30`
boxes.

### Note on the book

The explanatory paragraph in section `2.11.2` says "como mínimo 50 cajas", but equation `(2.108)`
shows `sum x_i >= 30` and the published AMPL solution (`15` tennis boxes and `15` sandals boxes)
is only feasible with `30`. In these notebooks we follow the equations and the published solution.


In [ ]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

try:
    from amplpy import AMPL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install amplpy in the active environment before running this notebook.") from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.eval(f"option solver {solver};")
    return ampl


BOXES = ["tennis", "boot", "heels", "sandals"]
COST = {"tennis": 2700, "boot": 2400, "heels": 640, "sandals": 625}
SHOE_PAIRS = {"tennis": 45, "boot": 30, "heels": 16, "sandals": 25}
UTILITY = {"tennis": 4500, "boot": 2550, "heels": 960, "sandals": 1250}
EXPECTED = {"tennis": 15, "boot": 0, "heels": 0, "sandals": 15, "total_utility": 86250, "total_cost": 49875}


@timed
def solve_shoe_problem_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set I ordered;
        param c {I} >= 0;
        param s {I} >= 0;
        param u {I} >= 0;
        var x {I} integer >= 0;

        maximize TotalUtility:
            sum {i in I} u[i] * x[i];

        subject to Budget:
            sum {i in I} c[i] * x[i] <= 50000;

        subject to MinimumBoxes:
            sum {i in I} x[i] >= 30;

        subject to SupplierBalance:
            sum {i in {"tennis", "boot"}} x[i] = sum {i in {"heels", "sandals"}} x[i];

        subject to MinimumPairs:
            sum {i in I} s[i] * x[i] >= 250;
        '''
    )
    ampl.set["I"] = BOXES
    ampl.param["c"] = COST
    ampl.param["s"] = SHOE_PAIRS
    ampl.param["u"] = UTILITY
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    result = {box: int(round(values[box])) for box in BOXES}
    result["total_utility"] = int(round(ampl.get_value("TotalUtility")))
    result["total_cost"] = sum(COST[box] * result[box] for box in BOXES)
    return result


## Printed AMPL Model


In [ ]:
result, elapsed = solve_shoe_problem_with_ampl()
print("AMPL solution:", result)
print(f"Elapsed time: {elapsed:.6f} seconds")
assert result == EXPECTED
print("\nThe AMPL result matches the printed equations and the published optimal solution.")
